##  주택청약 FAQ 시스템 챗봇 구현 - 문서 전처리 + RAG + Gradio ChatInterface

### 학습 목표

1. 텍스트 문서를 로드하고 구조화된 Q&A 쌍으로 파싱
2. LLM을 활용한 키워드 추출 및 요약 생성
3. Chroma 벡터 데이터베이스에 문서 임베딩 저장
4. MMR 검색 및 메타데이터 필터링 구현
5. RAG 체인 구성 및 문서 관련성 평가
6. Gradio를 사용한 대화형 챗봇 인터페이스 구현

---

### 사전 준비

**1. 환경 변수 설정**

`.env` 파일에 다음 내용을 추가하세요:
```
OPENAI_API_KEY=your-api-key-here
```

**2. 필수 패키지 설치**
```bash
pip install langchain-openai langchain-community langchain-core
pip install langchain-chroma chromadb
pip install python-dotenv gradio
```

**3. 데이터 파일**
- `data/housing_faq.txt` 파일 필요 (국토교통부 주택청약 FAQ 50개 Q&A)

# 환경 설정 및 준비

`(1) Env 환경변수`

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [6]:
import os
from glob import glob

from pprint import pprint
import json

`(3) LLM 설정`

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4.1-mini',      # 사용할 모델
    temperature=0.1,            # 낮은 값: 일관된 답변 (0.0~2.0)
    top_p=0.9,                  # 토큰 샘플링 확률 임계값 (0.0~1.0)
)

# **문서 전처리 파이프라인**

* 문서 전처리의 첫 단계는 데이터 정제로, 원본 문서에서 불필요한 요소(HTML 태그, 특수문자, 중복 내용 등)를 제거하고 텍스트를 표준화하는 과정입니다. 이는 검색 품질과 직결되는 중요한 단계입니다.

* 문서 청킹(Chunking)은 긴 문서를 의미 있는 작은 단위로 분할하는 과정으로, 문장 단위나 단락 단위로 나누되 문맥의 연속성을 유지하는 것이 핵심입니다. 이는 검색 정확도와 답변 생성의 품질에 직접적인 영향을 미칩니다.

* 임베딩(Embedding) 생성은 텍스트를 고차원의 벡터로 변환하는 과정으로, 문서의 의미적 특성을 수치화하여 효율적인 검색을 가능하게 합니다. 이때 사용되는 임베딩 모델의 선택이 검색 성능을 좌우하는 중요한 요소가 됩니다.

* 마지막으로 벡터 데이터베이스 색인화 단계에서는 생성된 임베딩을 효율적으로 저장하고 검색할 수 있는 구조로 변환합니다. 이는 대규모 문서 집합에서도 빠른 검색을 가능하게 하는 핵심 요소입니다.


### 1) 문서 로드

- 국토교통부 주택청약 FAQ에서 일부 내용(청약자격, 청약통장)을 발췌하여 재가공
- Q1 ~ Q50까지 모두 50개의 문답이 포함된 텍스트 파일

In [23]:
# 파일 경로 설정
faq_text_file = "data/housing_faq.txt"

# 파일 읽기 - 파이썬 내장 함수 사용
with open(faq_text_file, 'r') as f:
    faq_text = f.read()

# 파일 내용 확인
print(faq_text[:500])

Q1 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
A 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 
참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

Q2 해당 주택건설지역에 거주하고 있지 않다면 청약신청이 불가능한지?
A 해당 주택건설지역에 거주하고 있지 않더라도 청약가능지역에서 공급되는 주택에 청약신청이 가능하나, 같은 순위에서는 해당 주택건설지역의 거주자가 우선하여 주택을 공급받게 됩니다.
* 서울·인천·경기도 / 대전·세종·충남 / 충북 / 광주·전남 / 전북 / 대구·경북 / 부산·울산·경남 / 강원
다만, 수도권 대규모 택지개발지구 등에서 주택이 공급되는 경우 일정 비율의 


##### ***[실습] TextLoader를 사용하여, 텍스트 문서를 로드합니다.***

In [18]:
# 여기에 코드를 작성하세요.
from langchain_community.document_loaders import TextLoader

# TextLoader 클래스를 사용하여 FAQ 텍스트 파일을 로드
loader = TextLoader(faq_text_file)
docs = loader.load()
len(docs)
 
from langchain_core.documents import Document

def format_qa_pairs_with_summary(qa_pairs):
    """
    추출된 QA 쌍을 포맷팅하여 문서 객체로 변환
    """
    processed_docs = []
    for pair in qa_pairs:

        # 키워드와 요약 추출
        result = keyword_extractor.invoke(pair['question']+"\n\n"+pair['answer'])

        # 문서 객체 생성
        doc = Document(
            page_content=result.summary,
            metadata={
                'question_id': int(pair['number']),
                'question': pair['question'],
                'answer': pair['answer'],
                'keyword': result.keyword,
            }
        )
        processed_docs.append(doc)

    return processed_docs
from pprint import pprint
pprint(docs[0].metadata)

{'source': 'data/housing_faq.txt'}


In [24]:
from langchain_community.document_loaders import TextLoader

# 1. 파일 경로 설정 (파일명이 faq_data.txt라고 가정합니다)

try:
    # 2. TextLoader 클래스를 사용하여 로더 초기화
    loader = TextLoader(faq_text_file, encoding="utf-8") # 인코딩 설정을 넣어야 한글이 안 깨집니다.
    
    # 3. 문서 로드
    docs = loader.load()
    
    # 4. 결과 확인
    print(f"✅ 성공적으로 로드된 문서 수: {len(docs)}")
    print(f"📄 첫 번째 문서 내용 일부: {docs[0].page_content[:100]}...")

except FileNotFoundError:
    print(f"❌ 에러: '{faq_text_file}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
except Exception as e:
    print(f"❌ 에러 발생: {str(e)}")

✅ 성공적으로 로드된 문서 수: 1
📄 첫 번째 문서 내용 일부: Q1 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
A 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자...


In [25]:
# 문서 확인
print(docs[0].page_content[:500])

Q1 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
A 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 
참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

Q2 해당 주택건설지역에 거주하고 있지 않다면 청약신청이 불가능한지?
A 해당 주택건설지역에 거주하고 있지 않더라도 청약가능지역에서 공급되는 주택에 청약신청이 가능하나, 같은 순위에서는 해당 주택건설지역의 거주자가 우선하여 주택을 공급받게 됩니다.
* 서울·인천·경기도 / 대전·세종·충남 / 충북 / 광주·전남 / 전북 / 대구·경북 / 부산·울산·경남 / 강원
다만, 수도권 대규모 택지개발지구 등에서 주택이 공급되는 경우 일정 비율의 


In [26]:
# 문서 메타데이터 확인
docs[0].metadata

{'source': 'data/housing_faq.txt'}

### 2) 문서 전처리

`(1) 각 질문과 답변을 쌍으로 추출하여 정리 (정규표현식 활용)`

In [27]:
import re

def extract_qa_pairs(text):
    qa_pairs = []
    
    # 텍스트를 라인별로 분리하고 각 라인의 앞뒤 공백 제거
    lines = [line.strip() for line in text.split('\n')]
    current_question = None
    current_answer = []
    current_number = None
    in_answer = False
    
    for i, line in enumerate(lines):
        if not line:  # 빈 라인 처리
            if in_answer and current_answer and i + 1 < len(lines) and lines[i + 1].startswith('Q'):
                # 다음 질문이 시작되기 전 빈 줄이면 현재 QA 쌍 저장
                qa_pairs.append({
                    'number': current_number,
                    'question': current_question,
                    'answer': ' '.join(current_answer).strip()
                })
                in_answer = False
                current_answer = []
            continue
            
        # 새로운 질문 확인 (Q 다음에 숫자가 오는 패턴)
        q_match = re.match(r'Q(\d+)\s+(.*)', line)
        if q_match:
            # 이전 QA 쌍이 있으면 저장
            if current_question is not None and current_answer:
                qa_pairs.append({
                    'number': current_number,
                    'question': current_question,
                    'answer': ' '.join(current_answer).strip()
                })
            
            # 새로운 질문 시작
            current_number = int(q_match.group(1))
            current_question = q_match.group(2).strip().rstrip('?') + '?'  # 질문 마크 정규화
            current_answer = []
            in_answer = False
            
        # 답변 시작 확인
        elif line.startswith('A ') or (current_question and not current_answer and line):
            in_answer = True
            current_answer.append(line.lstrip('A '))
            
        # 기존 답변에 내용 추가
        elif current_question is not None and (in_answer or not line.startswith('Q')):
            if in_answer or (current_answer and not line.startswith('Q')):
                current_answer.append(line)
    
    # 마지막 QA 쌍 처리
    if current_question is not None and current_answer:
        qa_pairs.append({
            'number': current_number,
            'question': current_question,
            'answer': ' '.join(current_answer).strip()
        })
    
    # 번호 순서대로 정렬
    qa_pairs.sort(key=lambda x: x['number'])
    
    return qa_pairs

In [28]:
# QA 쌍 추출
qa_pairs = extract_qa_pairs(docs[0].page_content) 

print(f"추출된 QA 쌍 개수: {len(qa_pairs)}")
print(f"추출된 첫번째 QA: \n{qa_pairs[0]}")

추출된 QA 쌍 개수: 50
추출된 첫번째 QA: 
{'number': 1, 'question': '경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?', 'answer': '해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.'}


`(2) LLM으로 추가 정보를 추출`
- 텍스트에서 키워드와 핵심 개념을 추출하는 체인
- 메타데이터 or 본문(page_content)에 추가하여 검색에 활용    

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List

# 출력 형식 정의
class KeywordOutput(BaseModel):
    keyword: str = Field(description="텍스트에서 추출한 가장 중요한 키워드(법률용어, 주제 등))")
    summary: str = Field(description="텍스트의 간단한 요약")

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 텍스트 분석 전문가입니다. 
주어진 텍스트에서 중요한 키워드를 추출하고, 텍스트의 간단한 요약을 작성하는 것이 당신의 역할입니다.

## 추출 지침:
- 텍스트의 맥락을 고려하여 핵심 용어나 전문 용어를 추출합니다
- 주요 아이디어나 원리, 개념을 포함합니다
- 가장 중요한 키워드를 1개 추출합니다
- 요약은 1문장으로 간결하게 작성합니다

## 출력 형식:
- keyword: 가장 중요한 키워드 
- summary: 텍스트의 간단한 요약"""),
    
    ("user", "다음 텍스트를 분석해주세요:\n\n{input_text}")
])

# LCEL 체인 구성
llm_with_structure = llm.with_structured_output(KeywordOutput) 
keyword_extractor = prompt | llm_with_structure

# 텍스트 추출 테스트     
result = keyword_extractor.invoke(qa_pairs[0]['question']+qa_pairs[0]['answer'])
print("키워드:", result.keyword)
print("요약:", result.summary)

키워드: 해당 주택건설지역
요약: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역은 과천시 행정구역 전체를 의미한다.


`(3) QA 쌍을 문자열 포맷팅하고 문서 객체로 변환`

In [30]:
from langchain_core.documents import Document

def format_qa_pairs(qa_pairs):
    """
    추출된 QA 쌍을 포맷팅하여 문서 객체로 변환
    """
    processed_docs = []
    for pair in qa_pairs:

        # QA 쌍을 포맷팅
        formatted_output = (
            f"[{pair['number']}]\n"
            f"질문: {pair['question']}\n"
            f"답변: {pair['answer']}\n"
        )

        # 키워드와 요약 추출
        result = keyword_extractor.invoke(pair['question']+"\n\n"+pair['answer'])

        # 문서 객체 생성
        doc = Document(
            page_content=formatted_output,
            metadata={
                'question_id': int(pair['number']),
                'question': pair['question'],
                'answer': pair['answer'],
                'keyword': result.keyword,
                'summary': result.summary
            }
        )
        processed_docs.append(doc)

    return processed_docs


# QA 쌍 포맷팅
formatted_docs = format_qa_pairs(qa_pairs)
print(f"포맷팅된 문서 개수: {len(formatted_docs)}")

# 문서 확인
print(formatted_docs[0].page_content)
print("-" * 200)
# 문서 메타데이터 확인
pprint(formatted_docs[0].metadata)

포맷팅된 문서 개수: 50
[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
{'answer': '해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 '
           '말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 '
           '주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 '
           '전역이 해당 주택건설지역에 해당됩니다.',
 'keyword': '해당 주택건설지역',
 'question': '경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?',
 'question_id': 1,
 'summary': '경기도 과천시에서 공급되는 주택의 해당 주택건설지역은 과천시 행정구역 전체를 의미한다.'}


In [ ]:
import json
# 문서 저장
output_file = "data/housing_faq_formatted.json"

with open(output_file, 'w', encoding='utf-8-sig') as f:
    json.dump([doc.model_dump() for doc in formatted_docs], f, indent=2, ensure_ascii=False)  # 한글이 유니코드로 변환되지 않도록 설정
print(f"포맷팅된 문서를 {output_file}에 저장했습니다.")

포맷팅된 문서를 data/housing_faq_formatted.json에 저장했습니다.


##### ***[실습] 문서 객체를 포맷팅하여 구성합니다.***

- 요약문을 시맨틱 검색에 활용합니다. 다음 구조로 문서 객체를 생성합니다. 
    - page_content: 요약
    - metadata: 기타 정보




In [ ]:
# 여기에 코드를 작성하세요.

In [ ]:
from langchain_core.documents import Document

def format_qa_pairs_with_summary(qa_pairs):
    """
    추출된 QA 쌍을 포맷팅하여 문서 객체로 변환
    """
    processed_docs = []
    for pair in qa_pairs:

        # 키워드와 요약 추출
        result = None

        # 문서 객체 생성
        doc = None
        
        processed_docs.append(doc)

    return processed_docs


In [ ]:
# QA 쌍 포맷팅
summary_formatted_docs = format_qa_pairs_with_summary(qa_pairs) 
print(f"포맷팅된 문서 개수: {len(summary_formatted_docs)}")

# 문서 확인
print(summary_formatted_docs[0].page_content)
print("-" * 200)
# 문서 메타데이터 확인
pprint(summary_formatted_docs[0].metadata)

In [55]:
# 문서 저장
output_file = "data/housing_faq_formatted_with_summary.json"

with open(output_file, 'w', encoding='utf-8-sig') as f:
    json.dump([doc.model_dump() for doc in summary_formatted_docs], f, indent=2, ensure_ascii=False)  # 한글이 유니코드로 변환되지 않도록 설정
print(f"포맷팅된 문서를 {output_file}에 저장했습니다.")

NameError: name 'summary_formatted_docs' is not defined

In [36]:
from langchain_core.documents import Document

def format_qa_pairs_with_summary(qa_pairs):
    """
    추출된 QA 쌍을 포맷팅하여 LangChain Document 객체 리스트로 변환
    """
    processed_docs = []
    
    for pair in qa_pairs:
        # 1. 키워드 및 요약 추출 (미리 정의된 keyword_extractor 사용)
        # 질문과 답변을 합쳐서 컨텍스트로 전달합니다.
        combined_text = f"질문: {pair['question']}\n답변: {pair['answer']}"
        result = keyword_extractor.invoke(combined_text)

        # 2. 문서 객체 생성 (요구사항 반영)
        # page_content: 시맨틱 검색에 최적화된 요약문
        # metadata: 필터링 및 최종 답변 출력을 위한 상세 정보
        doc = Document(
            page_content=result.summary, 
            metadata={
                'question_id': int(pair.get('number', 0)),
                'original_question': pair['question'],
                'full_answer': pair['answer'],
                'keywords': result.keyword, # 리스트 형태일 경우 그대로 저장
                'source': 'housing_faq_2024'
            }
        )
        processed_docs.append(doc)

    return processed_docs

# 실행 및 결과 확인
formatted_docs = format_qa_pairs_with_summary(qa_pairs)

print(f"✅ 총 {len(formatted_docs)}개의 문서 객체가 생성되었습니다.")

# 첫 번째 문서의 구조 확인
from pprint import pprint
print("\n--- [첫 번째 문서 메타데이터 샘플] ---")
pprint(formatted_docs[0].metadata)

✅ 총 50개의 문서 객체가 생성되었습니다.

--- [첫 번째 문서 메타데이터 샘플] ---
{'full_answer': '해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 '
                '특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 '
                '과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, '
                '인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.',
 'keywords': '해당 주택건설지역',
 'original_question': '경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?',
 'question_id': 1,
 'source': 'housing_faq_2024'}


In [56]:
from langchain_core.documents import Document

def format_qa_pairs_with_summary(qa_pairs):
    """
    추출된 QA 쌍과 요약문을 결합하여 LangChain Document 객체 리스트 생성
    """
    summary_formatted_docs = []
    
    for pair in qa_pairs:
        # 1. 이전 단계에서 정의한 keyword_extractor를 사용하여 요약 및 키워드 추출
        # (이미 invoke 결과가 변수에 있다면 해당 값을 직접 사용해도 됩니다)
        combined_text = f"질문: {pair['question']}\n답변: {pair['answer']}"
        result = keyword_extractor.invoke(combined_text)

        # 2. 문서 객체 생성 (요구사항 반영)
        # page_content: 시맨틱 검색에 최적화된 '요약' 내용
        # metadata: 답변 생성을 위한 원문 질문, 전체 답변, 키워드 등 저장
        doc = Document(
            page_content=result.summary, 
            metadata={
                'question_id': int(pair.get('number', 0)),
                'question': pair['question'],
                'answer': pair['answer'],
                'keyword': result.keyword,
                'category': '주택청약_FAQ'
            }
        )
        summary_formatted_docs.append(doc)

    return summary_formatted_docs

# 실행: summary_formatted_docs 변수에 결과 저장
summary_formatted_docs = format_qa_pairs_with_summary(qa_pairs)

# 결과 확인: 첫 번째 문서의 구조 출력
from pprint import pprint
print(f"✅ 생성된 문서 개수: {len(summary_formatted_docs)}")
print("\n--- [첫 번째 문서 데이터 구조] ---")
pprint(summary_formatted_docs[0].dict())

✅ 생성된 문서 개수: 50

--- [첫 번째 문서 데이터 구조] ---
{'id': None,
 'metadata': {'answer': '해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 '
                        '없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 '
                        '공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 '
                        '주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 '
                        '해당됩니다.',
              'category': '주택청약_FAQ',
              'keyword': '해당 주택건설지역',
              'question': '경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?',
              'question_id': 1},
 'page_content': '경기도 과천시에서 공급되는 주택의 해당 주택건설지역은 과천시 행정구역 전체를 의미한다.',
 'type': 'Document'}


/var/folders/w1/ldkw4tpj0_18yf9m78_96mqr0000gn/T/ipykernel_60044/918150038.py:39: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  pprint(summary_formatted_docs[0].dict())


# 벡터 저장 

In [58]:
# 문서 로드
from langchain_core.documents import Document

output_file = "data/housing_faq_formatted.json"

with open(output_file, 'r', encoding='utf-8-sig') as f:
    formatted_docs = [Document(**doc) for doc in json.load(f)]

# 문서 확인
print(formatted_docs[0].page_content)

[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.



In [38]:
print(formatted_docs[0].metadata)

{'question_id': 1, 'question': '경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?', 'answer': '해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.', 'keyword': '해당 주택건설지역', 'summary': '경기도 과천시에서 공급되는 주택의 해당 주택건설지역은 과천시 행정구역 전체를 의미한다.'}


In [39]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 문서 벡터 저장
vector_store = Chroma.from_documents(  
    documents=formatted_docs,
    embedding=embeddings,
    collection_name="housing_faq_db",
    persist_directory="./chroma_db",
)

In [46]:
vector_store._collection.count()

50

In [47]:
# vector_store.delete_collection()

##### ***[실습] 요약 문서(summary_formatted_docs)를 벡터 스토어에 저장합니다.*** 

- OpenAI (text-embedding-3-small) 임베딩 모델 사용
- Chroma DB 사용

**힌트**:
1. `Chroma.from_documents()` 메소드 사용
2. `embedding` 파라미터에 OpenAIEmbeddings 인스턴스 전달
3. `collection_name`과 `persist_directory` 지정
4. `summary_formatted_docs` 변수 사용

In [59]:
# 여기에 코드를 작성하세요.
import json
import os

# 저장 경로 및 폴더 생성
output_file = "data/housing_faq_formatted_with_summary.json"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Document 객체 리스트를 JSON 저장 가능한 형태로 변환
# (이전 셀에서 생성한 formatted_docs 혹은 summary_formatted_docs 변수 사용)
serializable_data = [
    {"page_content": doc.page_content, "metadata": doc.metadata} 
    for doc in summary_formatted_docs  # 변수명이 다르면 수정하세요
]

with open(output_file, 'w', encoding='utf-8-sig') as f:
    json.dump(serializable_data, f, indent=2, ensure_ascii=False)

print(f"✅ 파일을 새로 저장했습니다: {output_file}")


✅ 파일을 새로 저장했습니다: data/housing_faq_formatted_with_summary.json


In [60]:
# 문서 로드
from langchain_core.documents import Document

output_file = "data/housing_faq_formatted_with_summary.json"

with open(output_file, 'r', encoding='utf-8-sig') as f:
    summary_formatted_docs = [Document(**doc) for doc in json.load(f)]

# 문서 확인
print(summary_formatted_docs[0].page_content)

경기도 과천시에서 공급되는 주택의 해당 주택건설지역은 과천시 행정구역 전체를 의미한다.


In [62]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 문서 벡터 저장
vector_store_summary = None
# 2. 저장 경로 설정 (로컬 디렉토리에 데이터 유지)
persist_directory = "./chroma_db_summary"
os.makedirs(persist_directory, exist_ok=True)

# 3. 문서 벡터 저장 (Chroma.from_documents 사용)
# summary_formatted_docs 변수가 이전 셀에서 성공적으로 생성되어 있어야 합니다.
vector_store_summary = Chroma.from_documents(
    documents=summary_formatted_docs, 
    embedding=embeddings,
    collection_name="housing_faq_summary",
    persist_directory=persist_directory
)

# 4. 저장된 컬렉션의 문서 개수 확인
print(f"✅ 저장된 문서 개수: {vector_store_summary._collection.count()}")

✅ 저장된 문서 개수: 50


# 문서 검색

In [63]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터 저장소 로드
vector_store = Chroma(
    collection_name="housing_faq_db",
    persist_directory="./chroma_db", 
    embedding_function=embeddings,
)

vector_store._collection.count()

50

##### ***[실습] 앞에서 저장한 요약문서 벡터 스토어를 로드합니다.*** 

- OpenAI (text-embedding-3-small) 임베딩 모델 사용
- Chroma DB 사용

**힌트**:
1. `Chroma()` 생성자 사용 (from_documents가 아님)
2. 저장 시 사용한 `collection_name`, `persist_directory` 동일하게 지정
3. `embedding_function` 파라미터에 OpenAIEmbeddings 전달

In [64]:
# 여기에 코드를 작성하세요.
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 1. 저장 시 사용했던 것과 동일한 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. 저장된 경로 및 콜렉션 이름 지정 (이전 단계와 동일해야 함)
persist_directory = "./chroma_db_summary"
collection_name = "housing_faq_summary"

# 3. Chroma 생성자를 사용하여 기존 DB 로드
# 힌트: embedding_function 파라미터를 사용합니다.
vector_store_summary = Chroma(
    collection_name=collection_name,
    persist_directory=persist_directory,
    embedding_function=embeddings
)

# 4. 로드된 문서 개수 확인 (정상적으로 로드되었는지 체크)
count = vector_store_summary._collection.count()
print(f"✅ 성공적으로 로드된 문서 개수: {count}")

# 벡터 저장소 로드
vector_store_summary = None

✅ 성공적으로 로드된 문서 개수: 50


In [65]:
# 검색기 생성 - 유사도 기반 상위 3개 문서 검색
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3},
)

# 테스트 질문
query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

--------------------------------------------------
해당 주택건설지역
1
[2]
질문: 해당 주택건설지역에 거주하고 있지 않다면 청약신청이 불가능한지?
답변: 해당 주택건설지역에 거주하고 있지 않더라도 청약가능지역에서 공급되는 주택에 청약신청이 가능하나, 같은 순위에서는 해당 주택건설지역의 거주자가 우선하여 주택을 공급받게 됩니다. * 서울·인천·경기도 / 대전·세종·충남 / 충북 / 광주·전남 / 전북 / 대구·경북 / 부산·울산·경남 / 강원 다만, 수도권 대규모 택지개발지구 등에서 주택이 공급되는 경우 일정 비율의 주택에 대해서는 해당 주택건설지역 거주자와 동등한 자격으로 주택을 공급받을 기회를 가지게 됩니다.

--------------------------------------------------
청약신청
2
[47]
질문: 무주택세대구성원이란?
답변: 무주택세대구성원이란 청약신청자 및 세대원 전원이 주택을 소유하고 있지 않은 세대의 구성원(세대주 포함)을 말합니다.

--------------------------------------------------
무주택세대구성원
47


#### MMR (Maximal Marginal Relevance)

MMR은 검색 결과의 **관련성**과 **다양성**을 동시에 고려하는 알고리즘입니다.

**주요 파라미터**:
- `fetch_k`: 초기 검색 문서 수 (유사도 기준)
- `k`: 최종 반환 문서 수
- `lambda_mult`: 다양성 가중치
  - `0.0`: 최대 다양성 (서로 다른 문서 우선)
  - `1.0`: 최대 관련성 (유사도만 고려)
  - `0.5`: 균형 (관련성과 다양성을 동등하게)

##### ***[실습] MMR 검색기를 정의합니다.***  

- 문서 벡터 스토어를 사용
- 10개의 문서를 가져와서, 다양성 기반으로 3개를 선택 (다양성은 중간 수준 적용)

**힌트**:
1. `vector_store.as_retriever()` 메소드 사용
2. `search_type="mmr"` 파라미터 지정
3. `search_kwargs`에 `fetch_k=10`, `k=3`, `lambda_mult=0.5` 설정

In [70]:
# 여기에 코드를 작성하세요.
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 1. 환경 설정 (이전에 설정했다면 생략 가능)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
persist_directory = "./chroma_db_summary"
collection_name = "housing_faq_summary"

# 2. 벡터 스토어 로드 (이 부분이 실행되어야 None이 아니게 됩니다)
vector_store_summary = Chroma(
    collection_name=collection_name,
    persist_directory=persist_directory,
    embedding_function=embeddings
)

# 3. 로드 확인 (숫자가 0보다 커야 합니다)
print(f"✅ 로드된 문서 개수: {vector_store_summary._collection.count()}")

# 4. 이제 MMR 검색기를 정의합니다
mmr_retriever = vector_store_summary.as_retriever(
    search_type="mmr",
    search_kwargs={
        'fetch_k': 10,
        'k': 3,
        'lambda_mult': 0.5
    }
)
print("✅ MMR 검색기가 성공적으로 정의되었습니다.")
# 앞에서 로드한 vector_store_summary 객체를 사용합니다.
# 1. vector_store.as_retriever() 메소드를 사용하여 검색기 정의
mmr_retriever = vector_store_summary.as_retriever(
    # 2. search_type="mmr" 파라미터 지정
    search_type="mmr",
    # 3. search_kwargs 설정
    # fetch_k: 후보군 10개 추출
    # k: 최종 선택 3개
    # lambda_mult: 다양성 가중치 (0에 가까울수록 다양성 중시, 1에 가까울수록 유사도 중시)
    search_kwargs={
        'fetch_k': 10,
        'k': 3,
        'lambda_mult': 0.5  # 중간 수준의 다양성 적용
    }
)

print("✅ MMR 검색기가 성공적으로 정의되었습니다.")
# 테스트 질문
query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = mmr_retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print(result.metadata['question'])
    print(result.metadata['answer'])
    print("=" * 50)

✅ 로드된 문서 개수: 50
✅ MMR 검색기가 성공적으로 정의되었습니다.
✅ MMR 검색기가 성공적으로 정의되었습니다.
경기도 과천시에서 공급되는 주택의 해당 주택건설지역은 과천시 행정구역 전체를 의미한다.
--------------------------------------------------
해당 주택건설지역
1
경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.
형제·자매가 동일 세대별 주민등록표에 등재되어도 무주택세대구성원으로 인정되지 않으며, 주택소유 여부 판단에 영향을 주지 않는다.
--------------------------------------------------
무주택세대구성원 인정
48
주택을 소유하고 있는 형제·자매의 동거인으로 같은 세대별 주민등록표에 등재된 경우 무주택세대구성원 인정이 가능한지?
직계존비속이 아닌 형제·자매는 청약신청자와 동일한 세대별 주민등록표에 등재되어 있어도 세대원으로 인정되지 않습니다. 따라서 형제·자매의 주택소유 및 청약 제한사항은 신청자의 주택소유 판단 시 영향을 미치지 않습니다.
서울시 102㎡ 이하 주택 청약을 위해서는 인천 거주자가 기존 400만원에서 600만원으로 부족분 200만원을 추가 예치해야 한다.
--------------------------------------------------
청약예금
39
인천광역시 거주자로 청약예금 400만원(전용면적 102㎡ 이하)에 가입한 자가 입주자 모집공고일 전 서울시로 이주한 경우 102㎡ 이하의 주택에 청약하려면?
서울의 경우 전용면적 102㎡ 이하 주택에 청약할 

### **[심화] 메타데이터 기반 필터링**

- Chroma 문서: https://docs.trychroma.com/docs/querying-collections/metadata-filtering

In [71]:
# 단일 필드 정확히 일치
retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"keyword": "해당 주택건설지역"}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

--------------------------------------------------
해당 주택건설지역
1


In [72]:
# $eq 연산자 사용 - 정확히 일치

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"keyword": {"$eq": "해당 주택건설지역"}}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

--------------------------------------------------
해당 주택건설지역
1


In [73]:
# $ne (Not Equal) 연산자 사용 - 정확히 일치하지 않는 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"keyword": {"$ne": "해당 주택건설지역"}}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

[2]
질문: 해당 주택건설지역에 거주하고 있지 않다면 청약신청이 불가능한지?
답변: 해당 주택건설지역에 거주하고 있지 않더라도 청약가능지역에서 공급되는 주택에 청약신청이 가능하나, 같은 순위에서는 해당 주택건설지역의 거주자가 우선하여 주택을 공급받게 됩니다. * 서울·인천·경기도 / 대전·세종·충남 / 충북 / 광주·전남 / 전북 / 대구·경북 / 부산·울산·경남 / 강원 다만, 수도권 대규모 택지개발지구 등에서 주택이 공급되는 경우 일정 비율의 주택에 대해서는 해당 주택건설지역 거주자와 동등한 자격으로 주택을 공급받을 기회를 가지게 됩니다.

--------------------------------------------------
청약신청
2
[47]
질문: 무주택세대구성원이란?
답변: 무주택세대구성원이란 청약신청자 및 세대원 전원이 주택을 소유하고 있지 않은 세대의 구성원(세대주 포함)을 말합니다.

--------------------------------------------------
무주택세대구성원
47
[7]
질문: 행정중심복합도시예정지역에서 공급하는 주택의 경우 공급비율 및 대상은?
답변: 행정중심복합도시 예정지역에서 공급하는 주택의 경우 「주택공급에 관한 규칙」 제34조에 따라 해당 주택건설지역 거주자에게 행정중심복합도시건설청장이 정하여 고시하는 비율(현행 60%)을 우선공급하고 있으며, 이후 남은 물량은 「주택공급에 관한 규칙」 제4조제1항제3호 가목에 따라 해당 주택 건설지역에 거주하지 않는 자도 공급대상에 포함하여 공급하고 있습니다.

--------------------------------------------------
행정중심복합도시 주택 공급 비율
7
[6]
질문: 「주택공급에 관한 규칙」 제34조에 따른 대규모택지개발지구에서 주택이 공급되는 경우 일반공급 뿐만 아니라 특별공급 물량도 공급비율에 따라 배정되는지?
답변: 공급규칙 제34조가 적용되는 지역에 주택을 공급하는 경우 특별공급 물량 또한 그 공급 비율에 따라 배정하여야

In [74]:
# $in 연산자로 여러 값 중 일치하는 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"keyword": {"$in": ["해당 주택건설지역", "청약예금"]}}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

--------------------------------------------------
해당 주택건설지역
1
[39]
질문: 인천광역시 거주자로 청약예금 400만원(전용면적 102㎡ 이하)에 가입한 자가 입주자 모집공고일 전 서울시로 이주한 경우 102㎡ 이하의 주택에 청약하려면?
답변: 서울의 경우 전용면적 102㎡ 이하 주택에 청약할 수 있는 예치금액은 600만원이기 때문에 청약접수 당일까지 부족금액인 200만원을 추가로 예치하여야만 102㎡ 이하의 주택에 청약이 가능

--------------------------------------------------
청약예금
39


In [75]:
# 숫자 범위 검색 ($gt, $gte, $lt, $lte) - question_id가 10 이상인 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"question_id": {"$gte": 10}}},
)

query = "무주택자 기준은 무엇인가요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

[50]
질문: 무주택자인 아내가 유주택자인 남편과 주민등록표상 분리되어 친정부모의 세대별 주민등록표에 등재되어 있는 경우, 무주택자인 친정부모는 무주택세대구성원 자격이 인정되는지?
답변: 친정부모의 세대원 범위에 세대분리된 직계비속의 배우자(사위)는 포함되지 않으므로 사위가 주택을 소유하고 있다 하더라도 무주택세대구성원으로 인정됩니다.

--------------------------------------------------
무주택세대구성원
50
[47]
질문: 무주택세대구성원이란?
답변: 무주택세대구성원이란 청약신청자 및 세대원 전원이 주택을 소유하고 있지 않은 세대의 구성원(세대주 포함)을 말합니다.

--------------------------------------------------
무주택세대구성원
47
[31]
질문: 청년주택드림청약통장의 소득공제 혜택은 기존 주택청약종합저축과 동일한가요?
답변: 현재 주택청약종합저축에서 제공하는 소득공제 조건(조세특례제한법 제87조)을 그대로 적용받게 되며, 연소득 7천만원 이하 무주택세대주로 무주택확인서를 제출하는 경우 연간 납입액 300만원 한도로 40%까지 소득공제가 가능합니다.

--------------------------------------------------
소득공제
31
[36]
질문: 청년주택드림청약통장 가입(혹은 전환신규) 당시 무주택자 였는데 이후 주택을 소유하게 된 경우 우대이율 및 비과세 적용을 받을 수 있나요?
답변: 청년주택드림통장의 우대이율요건과 비과세 요건은 동일하지 않습니다. 비과세 요건 중 ‘주택을 소유하지 않은 세대의 세대주’ 자격은 가입 당시 기준입니다. 따라서 이후 주택을 소유하더라도 비과세 적용 가능합니다. 다만, 우대이율의 경우 최초로 주택을 소유하게 된 년도의 직전년도 말까지만 우대이율을 적용받습니다. * 예시) ’21년에 가입 후 ’23년에 주택 구입 시 : ’22년 말까지 우대이율 적용

-------------------------------------

In [76]:
# $and로 여러 조건 조합 - keyword가 "주택건설지역"이고 question_id가 10 미만인 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"$and": [
        {"keyword": "해당 주택건설지역"}, 
        {"question_id": {"$lt": 10}}
    ]}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

--------------------------------------------------
해당 주택건설지역
1


In [77]:
# $or로 여러 조건 중 하나 일치하는 문서 검색 - keyword가 "주택건설지역"이거나 question_id가 10 이상인 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"$or": [
        {"keyword": "해당 주택건설지역"}, 
        {"question_id": {"$gte": 10}}
    ]}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

--------------------------------------------------
해당 주택건설지역
1
[47]
질문: 무주택세대구성원이란?
답변: 무주택세대구성원이란 청약신청자 및 세대원 전원이 주택을 소유하고 있지 않은 세대의 구성원(세대주 포함)을 말합니다.

--------------------------------------------------
무주택세대구성원
47
[50]
질문: 무주택자인 아내가 유주택자인 남편과 주민등록표상 분리되어 친정부모의 세대별 주민등록표에 등재되어 있는 경우, 무주택자인 친정부모는 무주택세대구성원 자격이 인정되는지?
답변: 친정부모의 세대원 범위에 세대분리된 직계비속의 배우자(사위)는 포함되지 않으므로 사위가 주택을 소유하고 있다 하더라도 무주택세대구성원으로 인정됩니다.

--------------------------------------------------
무주택세대구성원
50
[38]
질문: 주민등록 거주지가 인천시인 청약신청자가 서울에서 분양하는 전용면적 85㎡이하의 민영주택을 청약코자 할 때 예치기준금액은?
답변: 주택공급에 관한 규칙 [별표2]에서 민영주택 청약 예치기준금액을 규정하고 있으며, 지역은 입주자모집공고일 현재 청약신청자의 주민등록표등초본상 거주지 기준, 예치 기준금액은 청약하고자 하는 주택의 평형을 기준으로 결정하여야 함. 따라서, 청약자가 인천시(그 밖의 광역시)에 거주하고 있고, 85㎡ 이하의 평형에 청약하는 경우 예치 기준금액

In [78]:
# 정규식 패턴 매칭 - page_content 본문에 "주택건설지역"이 포함된 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={'where_document': {'$contains': '해당 주택건설지역'}},
)

query = "수원시의 '해당 주택건설지역'은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

--------------------------------------------------
해당 주택건설지역
1
[2]
질문: 해당 주택건설지역에 거주하고 있지 않다면 청약신청이 불가능한지?
답변: 해당 주택건설지역에 거주하고 있지 않더라도 청약가능지역에서 공급되는 주택에 청약신청이 가능하나, 같은 순위에서는 해당 주택건설지역의 거주자가 우선하여 주택을 공급받게 됩니다. * 서울·인천·경기도 / 대전·세종·충남 / 충북 / 광주·전남 / 전북 / 대구·경북 / 부산·울산·경남 / 강원 다만, 수도권 대규모 택지개발지구 등에서 주택이 공급되는 경우 일정 비율의 주택에 대해서는 해당 주택건설지역 거주자와 동등한 자격으로 주택을 공급받을 기회를 가지게 됩니다.

--------------------------------------------------
청약신청
2
[7]
질문: 행정중심복합도시예정지역에서 공급하는 주택의 경우 공급비율 및 대상은?
답변: 행정중심복합도시 예정지역에서 공급하는 주택의 경우 「주택공급에 관한 규칙」 제34조에 따라 해당 주택건설지역 거주자에게 행정중심복합도시건설청장이 정하여 고시하는 비율(현행 60%)을 우선공급하고 있으며, 이후 남은 물량은 「주택공급에 관한 규칙」 제4조제1항제3호 가목에 따라 해당 주택 건설지역에 거주하지 않는 자도 공급대상에 포함하여 공급하고 있습니다.

--------------------------------------------------
행정중심복합도

##### ***[실습] 메타데이터 필터링 조건을 적용하는 실습을 수행합니다..*** 

- 요약 문서 벡터 스토어 기반 MMR 검색기에 적용 

In [79]:
# 여기에 코드를 작성하세요.
# 1. 필터링 조건 정의 (예: question_id가 10번 이하인 데이터만 검색)
# 실제 데이터에 맞춰 조건을 수정해 보세요.
search_filter = {
    "question_id": {"$lte": 10}  # ID가 10보다 작거나 같은(Less Than or Equal) 경우
}

# 2. 필터링 조건이 포함된 MMR 검색기 생성
mmr_filter_retriever = vector_store_summary.as_retriever(
    search_type="mmr",
    search_kwargs={
        'fetch_k': 10,       # 후보군 10개 추출
        'k': 3,              # 최종 선택 3개
        'lambda_mult': 0.5,  # 다양성 가중치
        'filter': search_filter # 메타데이터 필터 적용
    }
)

print("✅ 메타데이터 필터가 포함된 MMR 검색기가 정의되었습니다.")

✅ 메타데이터 필터가 포함된 MMR 검색기가 정의되었습니다.


### 메타데이터 필터 LLM 추출

In [80]:
from pydantic import BaseModel, Field
from typing import Optional, Literal


class MetadataFilter(BaseModel):
    """Chroma DB 메타데이터 필터 조건"""

    # 키워드 필터
    keyword: Optional[str] = Field(default=None, description="검색할 키워드")
    keyword_operator: Optional[Literal["$eq", "$ne"]] = Field(
        default=None, description="키워드 비교 연산자"
    )

    # 질문 ID 범위 (하한)
    question_id_min: Optional[int] = Field(default=None, description="질문 ID 최소값")
    question_id_min_operator: Optional[Literal["$gt", "$gte"]] = Field(
        default=None, description="최소값 연산자 ($gt: 초과, $gte: 이상)"
    )

    # 질문 ID 범위 (상한)
    question_id_max: Optional[int] = Field(default=None, description="질문 ID 최대값")
    question_id_max_operator: Optional[Literal["$lt", "$lte"]] = Field(
        default=None, description="최대값 연산자 ($lt: 미만, $lte: 이하)"
    )

    # 논리 연산자
    logical_operator: Optional[Literal["$and", "$or"]] = Field(
        default="$and", description="복합 조건 결합 연산자"
    )

In [81]:
from langchain_core.prompts import ChatPromptTemplate

# 시스템 프롬프트
METADATA_FILTER_SYSTEM_PROMPT = """사용자 쿼리에서 Chroma DB 검색 필터 조건을 추출합니다.

## 추출 규칙

### 키워드 (keyword)
- 특정 단어/주제 검색 시 해당 키워드 추출
- keyword_operator: 일반적으로 "$eq" 사용

### 질문 ID 범위
- "N번 이상": question_id_min=N, question_id_min_operator="$gte"
- "N번 초과": question_id_min=N, question_id_min_operator="$gt"
- "N번 이하": question_id_max=N, question_id_max_operator="$lte"
- "N번 미만": question_id_max=N, question_id_max_operator="$lt"
- "N~M번 사이": 최소값과 최대값 모두 설정

### 논리 연산자
- 모든 조건 만족: "$and" (기본값)
- 하나라도 만족: "$or"

## 예시

1. 키워드만: "주택건설 관련 문서"
   → keyword="주택건설", keyword_operator="$eq"

2. ID 범위: "10번 이상 20번 이하"
   → question_id_min=10, question_id_min_operator="$gte",
     question_id_max=20, question_id_max_operator="$lte"

3. 복합 조건: "청약통장 관련 40~50번 문서"
   → keyword="청약통장", keyword_operator="$eq",
     question_id_min=40, question_id_min_operator="$gte",
     question_id_max=50, question_id_max_operator="$lte",
     logical_operator="$and"

해당 정보가 없으면 null 반환.
"""

# 메타데이터 추출 체인 구성
metadata_extraction_chain = (
    ChatPromptTemplate.from_messages([
        ("system", METADATA_FILTER_SYSTEM_PROMPT),
        ("human", "{query}")
    ])
    | llm.with_structured_output(MetadataFilter)
)

# 테스트: 필터 조건 추출
query = "'해당 주택건설지역' 관련 문서를 10번 이하인 문서중에서 검색해주세요"
filter_params = metadata_extraction_chain.invoke({"query": query})

print("추출된 필터 파라미터:")
print(filter_params.model_dump())

추출된 필터 파라미터:
{'keyword': '해당 주택건설지역', 'keyword_operator': '$eq', 'question_id_min': None, 'question_id_min_operator': None, 'question_id_max': 10, 'question_id_max_operator': '$lte', 'logical_operator': '$and'}


In [82]:
def build_chroma_filter(filter_params: MetadataFilter) -> dict:
    """MetadataFilter를 Chroma DB 필터 딕셔너리로 변환
    
    Args:
        filter_params: MetadataFilter 인스턴스
        
    Returns:
        Chroma DB where 절에 사용할 필터 딕셔너리
    """
    conditions = []

    # 키워드 조건
    if filter_params.keyword and filter_params.keyword_operator:
        conditions.append({
            "keyword": {filter_params.keyword_operator: filter_params.keyword}
        })

    # 질문 ID 최소값 조건
    if filter_params.question_id_min is not None and filter_params.question_id_min_operator:
        conditions.append({
            "question_id": {filter_params.question_id_min_operator: filter_params.question_id_min}
        })

    # 질문 ID 최대값 조건
    if filter_params.question_id_max is not None and filter_params.question_id_max_operator:
        conditions.append({
            "question_id": {filter_params.question_id_max_operator: filter_params.question_id_max}
        })

    # 조건 개수에 따른 필터 구성
    if len(conditions) == 0:
        return {}
    elif len(conditions) == 1:
        return conditions[0]
    else:
        logical_op = filter_params.logical_operator or "$and"
        return {logical_op: conditions}


# 테스트: 필터 딕셔너리 생성
filter_dict = build_chroma_filter(filter_params)
print("생성된 Chroma 필터:")
print(filter_dict)

생성된 Chroma 필터:
{'$and': [{'keyword': {'$eq': '해당 주택건설지역'}}, {'question_id': {'$lte': 10}}]}


In [83]:
from langchain_core.runnables import chain


@chain
def metadata_filter_retriever(query: str):
    """메타데이터 필터를 추출하고 검색 수행
    
    Args:
        query: 사용자 검색 쿼리
        
    Returns:
        검색된 문서 리스트
    """
    # 1. 필터 조건 추출
    filter_params = metadata_extraction_chain.invoke({"query": query})

    # 2. Chroma 필터로 변환
    filter_dict = build_chroma_filter(filter_params)
    print(f"추출된 필터: {filter_dict}")

    # 3. 검색 실행
    retriever = vector_store.as_retriever(
        search_kwargs={"filter": filter_dict} if filter_dict else {}
    )
    return retriever.invoke(query)


# 테스트 실행
query = "청약통장 관련 문서를 40번과 50번 사이의 문서 중에서 검색해주세요"
results = metadata_filter_retriever.invoke(query)

print(f"\n검색된 문서 수: {len(results)}")
for i, result in enumerate(results, 1):
    print(f"\n--- 문서 {i} ---")
    print(f"내용: {result.page_content[:100]}...")
    print(f"키워드: {result.metadata.get('keyword', 'N/A')}")
    print(f"질문 ID: {result.metadata.get('question_id', 'N/A')}")

추출된 필터: {'$and': [{'keyword': {'$eq': '청약통장'}}, {'question_id': {'$gte': 40}}, {'question_id': {'$lte': 50}}]}

검색된 문서 수: 1

--- 문서 1 ---
내용: [43]
질문: 청약통장이 압류 등으로 사용에 제한이 있거나, 청약통장 담보대출을 받고 있는 경우에도 해당 통장을 이용하여 당첨이 가능한지?
답변: 압류 등의 상태라고 하더라도 청...
키워드: 청약통장
질문 ID: 43


In [84]:
# 다양한 쿼리 테스트
test_queries = [
    "청약통장 관련 문서 찾아줘",        # 키워드만
    "질문 ID 10번 이상인 문서",         # ID 하한만
    "20번 이하 문서만 보여줘",          # ID 상한만
    "10번에서 30번 사이 문서",          # ID 범위
    "'해당 주택건설지역' 관련 10번 이하 문서",  # 복합
    "청약통장 관련 40~50번 문서",       # 복합 + 범위
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"쿼리: {query}")
    print('='*60)
    
    # 필터 추출 및 검색
    filter_params = metadata_extraction_chain.invoke({"query": query})
    filter_dict = build_chroma_filter(filter_params)
    print(f"필터: {filter_dict}")
    
    # 검색 실행
    results = metadata_filter_retriever.invoke(query)
    print(f"검색 결과: {len(results)}건")
    
    if results:
        print(f"첫 번째 문서 - 키워드: {results[0].metadata.get('keyword')}, ID: {results[0].metadata.get('question_id')}")


쿼리: 청약통장 관련 문서 찾아줘
필터: {'keyword': {'$eq': '청약통장'}}
추출된 필터: {'keyword': {'$eq': '청약통장'}}
검색 결과: 1건
첫 번째 문서 - 키워드: 청약통장, ID: 43

쿼리: 질문 ID 10번 이상인 문서
필터: {'question_id': {'$gte': 10}}
추출된 필터: {'question_id': {'$gte': 10}}
검색 결과: 4건
첫 번째 문서 - 키워드: 거주기간 인정, ID: 20

쿼리: 20번 이하 문서만 보여줘
필터: {'question_id': {'$lte': 20}}
추출된 필터: {'question_id': {'$lte': 20}}
검색 결과: 4건
첫 번째 문서 - 키워드: 거주기간 인정, ID: 20

쿼리: 10번에서 30번 사이 문서
필터: {'$and': [{'question_id': {'$gte': 10}}, {'question_id': {'$lte': 30}}]}
추출된 필터: {'$and': [{'question_id': {'$gte': 10}}, {'question_id': {'$lte': 30}}]}
검색 결과: 4건
첫 번째 문서 - 키워드: 예치금 정정 불가, ID: 23

쿼리: '해당 주택건설지역' 관련 10번 이하 문서
필터: {'$and': [{'keyword': {'$eq': '해당 주택건설지역'}}, {'question_id': {'$lte': 10}}]}
추출된 필터: {'$and': [{'keyword': {'$eq': '해당 주택건설지역'}}, {'question_id': {'$lte': 10}}]}
검색 결과: 1건
첫 번째 문서 - 키워드: 해당 주택건설지역, ID: 1

쿼리: 청약통장 관련 40~50번 문서
필터: {'$and': [{'keyword': {'$eq': '청약통장'}}, {'question_id': {'$gte': 40}}, {'question_id': {'$lte': 50}}]}
추출된 필터: {'$and

##### ***[실습] 메타데이터 필터링 체인의 성능을 개선합니다.*** 

- (예시)
    - 추가 예시 제공을 통해 필터링 적용 법위 확대

In [ ]:
# 여기에 코드를 작성하세요.
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain_openai import ChatOpenAI

# 1. 메타데이터 구조 정의 (LLM에게 각 필드가 무엇인지 설명)
metadata_field_info = [
    AttributeInfo(
        name="category",
        description="FAQ의 카테고리 (예: '신혼부부', '다자녀', '노부모부양', '일반공급')",
        type="string",
    ),
    AttributeInfo(
        name="question_id",
        description="질문의 고유 번호",
        type="integer",
    ),
    AttributeInfo(
        name="keyword",
        description="질문과 관련된 핵심 키워드",
        type="string",
    ),
]

document_content_description = "주택 청약 관련 FAQ 요약 및 답변 정보"

# 2. LLM 설정 (필터 추출을 담당할 브레인)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. 필터링 체인 성능 개선 (SelfQueryRetriever 정의)
# 추가 예시를 통해 필터링 범위를 확대합니다.
retriever = SelfQueryRetriever.from_llm(
    llm,
    vector_store_summary, # 이전에 생성한 Chroma 벡터 스토어
    document_content_description,
    metadata_field_info,
    verbose=True, # 필터가 어떻게 생성되는지 로그를 확인합니다.
    search_kwargs={"k": 3} # 최종적으로 3개의 문서만 가져옴
)

print("✅ 필터링 성능이 개선된 Self-Querying 검색기가 정의되었습니다.")

# RAG Chain

### 1) 참조 문서 없이 직접 답변을 생성

In [85]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Prompt
template = '''Answer the question based only on the following context.

[Context]
{context}

[Question]
{question}

[Answer (in 한국어)]
'''

prompt = ChatPromptTemplate.from_template(template)


# 문서 포맷팅
def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])


# 검색기 생성 - 유사도 기반 상위 3개 문서 검색
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3},
)


# Chain 구성
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Chain 실행
query = "수원시의 주택건설지역은 어디에 해당하나요?"
rag_chain.invoke(query)

'수원시는 경기도에 속하는 시이므로, 수원시에서 공급되는 주택의 해당 주택건설지역은 수원시 행정구역 전체에 해당합니다.'

### 2) 참조 문서를 답변과 함께 반환

`(1) 문서와 포맷팅된 컨텍스트를 함께 반환하는 함수`


In [86]:
from typing import Dict

def get_context_and_docs(question: str) -> Dict:
    """문서와 포맷팅된 컨텍스트를 함께 반환
    
    Args:
        question: 검색할 질문

    Returns:
        Dict: 문서와 포맷팅된 컨텍스트, 검색된 문서 리스트
    """

    # 검색 결과 가져오기
    docs = retriever.invoke(question)
    return {
        "question": question,  # 질문
        "context": format_docs(docs),   # 문서 포맷팅된 컨텍스트
        "source_documents": docs   # 검색된 문서 리스트
    }

`(2) 컨텍스트와 질문을 입력으로 받아 답변을 생성하는 함수`

In [87]:
from langchain_core.output_parsers import StrOutputParser

def prompt_and_generate_answer(input_data: Dict) -> Dict:
    """컨텍스트와 질문을 입력으로 받아 답변을 생성

    Args:
        input_data (Dict): 컨텍스트와 질문이 포함된 딕셔너리

    Returns:
        Dict: 생성된 답변과 소스 문서 정보가 포함된 딕셔너리
    """

    # LCEL 체인 구성 (StrOutputParser 사용)
    answer_chain = prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({
        "question":input_data["question"],
        "context":input_data["context"]
    })

    return {
        "answer": answer,  # 생성된 답변 (answer_chain 결과)
        "source_documents": input_data["source_documents"]  # 소스 문서 정보 (input_data에서 가져옴)
    }

`(3) RAG 체인 구성`

In [88]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from operator import itemgetter

# Chain 구성
rag_chain = (
    RunnableLambda(get_context_and_docs) |  # 문서와 컨텍스트 가져오기 
    {
        'response': RunnableLambda(prompt_and_generate_answer), # 답변 생성
        'question': itemgetter("question"),  # 질문 그대로 전달
        "source_documents": itemgetter("source_documents")   # 소스 반환
    }
)

In [89]:
# Chain 실행
query = "수원시의 주택건설지역은 어디에 해당하나요?"
result = rag_chain.invoke(query)

# 결과 출력
print("답변:", result["response"]["answer"])
print("\n참조 문서:")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n문서 {i}:")
    print(f"내용: {doc.page_content}")

답변: 수원시의 주택건설지역은 수원시 행정구역 전체에 해당합니다.

참조 문서:

문서 1:
내용: [1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.


문서 2:
내용: [2]
질문: 해당 주택건설지역에 거주하고 있지 않다면 청약신청이 불가능한지?
답변: 해당 주택건설지역에 거주하고 있지 않더라도 청약가능지역에서 공급되는 주택에 청약신청이 가능하나, 같은 순위에서는 해당 주택건설지역의 거주자가 우선하여 주택을 공급받게 됩니다. * 서울·인천·경기도 / 대전·세종·충남 / 충북 / 광주·전남 / 전북 / 대구·경북 / 부산·울산·경남 / 강원 다만, 수도권 대규모 택지개발지구 등에서 주택이 공급되는 경우 일정 비율의 주택에 대해서는 해당 주택건설지역 거주자와 동등한 자격으로 주택을 공급받을 기회를 가지게 됩니다.


문서 3:
내용: [47]
질문: 무주택세대구성원이란?
답변: 무주택세대구성원이란 청약신청자 및 세대원 전원이 주택을 소유하고 있지 않은 세대의 구성원(세대주 포함)을 말합니다.



### 3) 검색 문서 관련성 평가

`(1) 검색 문서와 질문 간의 관련성을 평가`


In [90]:
# 검색 문서의 질문 관련성 평가

prompt = ChatPromptTemplate.from_messages([
    ("system", """주어진 컨텍스트가 질문에 답변하는데 필요한 정보를 포함하고 있는지 논리적으로 평가하세요.
단계적으로 진행하며, 평가결과에 대한 검증을 수행하세요.

다음 기준 중 하나 이상을 충족할 경우 'Yes'로 답변하고, 모두 충족하지 못하면 'No'로 답변하세요:

1. 컨텍스트가 질문에 답변하는데 필요한 정보를 직접적으로 포함하고 있는가?
2. 컨텍스트의 정보로부터 답변에 필요한 내용을 논리적으로 추론할 수 있는가?
3. 컨텍스트의 정보가 질문에 대한 답변을 제공할 수 있는가?

'Yes' 또는 'No'로만 답변하세요."""),
    ("human", """[컨텍스트]
{context}

[질문]
{question}""")
])

chain = prompt | llm | StrOutputParser()    # gpt-4.1-mini 모델 사용

for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n문서 {i}:")
    print(f"내용: {doc.page_content}")
    relevance = chain.invoke({
        "context": doc.page_content,
        "question": query
    }).lower()

    print(f"평가 결과: {relevance}")


문서 1:
내용: [1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

평가 결과: yes

문서 2:
내용: [2]
질문: 해당 주택건설지역에 거주하고 있지 않다면 청약신청이 불가능한지?
답변: 해당 주택건설지역에 거주하고 있지 않더라도 청약가능지역에서 공급되는 주택에 청약신청이 가능하나, 같은 순위에서는 해당 주택건설지역의 거주자가 우선하여 주택을 공급받게 됩니다. * 서울·인천·경기도 / 대전·세종·충남 / 충북 / 광주·전남 / 전북 / 대구·경북 / 부산·울산·경남 / 강원 다만, 수도권 대규모 택지개발지구 등에서 주택이 공급되는 경우 일정 비율의 주택에 대해서는 해당 주택건설지역 거주자와 동등한 자격으로 주택을 공급받을 기회를 가지게 됩니다.

평가 결과: no

문서 3:
내용: [47]
질문: 무주택세대구성원이란?
답변: 무주택세대구성원이란 청약신청자 및 세대원 전원이 주택을 소유하고 있지 않은 세대의 구성원(세대주 포함)을 말합니다.

평가 결과: no


In [91]:
# gpt-4.1 모델 사용

llm_gpt4o = ChatOpenAI(
    model='gpt-4.1',          # 사용할 모델
    temperature=0.1,          # 낮은 값: 일관된 답변 (0.0~2.0)
    top_p=0.9,                # 토큰 샘플링 확률 임계값 (0.0~1.0)
)

chain = prompt | llm_gpt4o | StrOutputParser()    # gpt-4.1 모델 사용

for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n문서 {i}:")
    print(f"내용: {doc.page_content}")
    relevance = chain.invoke({
        "context": doc.page_content,
        "question": query
    }).lower()

    print(f"평가 결과: {relevance}")


문서 1:
내용: [1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.

평가 결과: yes

문서 2:
내용: [2]
질문: 해당 주택건설지역에 거주하고 있지 않다면 청약신청이 불가능한지?
답변: 해당 주택건설지역에 거주하고 있지 않더라도 청약가능지역에서 공급되는 주택에 청약신청이 가능하나, 같은 순위에서는 해당 주택건설지역의 거주자가 우선하여 주택을 공급받게 됩니다. * 서울·인천·경기도 / 대전·세종·충남 / 충북 / 광주·전남 / 전북 / 대구·경북 / 부산·울산·경남 / 강원 다만, 수도권 대규모 택지개발지구 등에서 주택이 공급되는 경우 일정 비율의 주택에 대해서는 해당 주택건설지역 거주자와 동등한 자격으로 주택을 공급받을 기회를 가지게 됩니다.

평가 결과: no

문서 3:
내용: [47]
질문: 무주택세대구성원이란?
답변: 무주택세대구성원이란 청약신청자 및 세대원 전원이 주택을 소유하고 있지 않은 세대의 구성원(세대주 포함)을 말합니다.

평가 결과: no


##### ***[실습] 문서 관련성 평가 체인을 구조화 출력으로 구현합니다.*** 

- pydantic schema 사용
- with_structured_output 함수 사용

In [94]:
# 여기에 코드를 작성하세요.
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# 1. 출력 구조 정의 (Pydantic Schema)
class GradeDocuments(BaseModel):
    """문서의 질문 관련성 평가 결과"""
    
    binary_score: str = Field(
        description="문서가 질문과 관련이 있는지 여부. 관련이 있으면 'yes', 없으면 'no'로 답변하세요."
    )
    reason: str = Field(
        description="관련성이 있다고 판단한 이유 혹은 부족한 점에 대한 짧은 설명"
    )

# 2. LLM 및 구조화된 출력 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# with_structured_output을 사용하여 LLM이 항상 GradeDocuments 형식으로만 답하게 강제함
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# 3. 프롬프트 구성
system_prompt = """당신은 검색된 문서가 사용자의 질문과 관련이 있는지 평가하는 전문가입니다.
만약 문서에 질문에 답하는 데 도움이 되는 키워드나 내용이 포함되어 있다면 'yes'로 평가하세요.
엄격하게 평가할 필요는 없으며, 조금이라도 관련이 있다면 'yes'를 선택하세요."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "사용자 질문: {question}\n\n검색된 문서 내용: {context}"),
    ]
)

# 4. 평가 체인 생성
grader_chain = grade_prompt | structured_llm_grader

print("✅ 구조화된 문서 관련성 평가 체인이 준비되었습니다.")

✅ 구조화된 문서 관련성 평가 체인이 준비되었습니다.


## Gradio 챗봇 인터페이스 (RAG 시스템 클래스 구현)

다음 클래스는 RAG 시스템의 전체 파이프라인을 캡슐화합니다.

**주요 메서드**:
- `_format_docs()`: 검색된 문서를 컨텍스트 문자열로 변환
- `_format_source_documents()`: 참조 문서를 사용자 친화적 형식으로 포맷
- `_evaluate_relevance()`: 검색된 문서의 질문 관련성 평가
- `_generate_answer()`: 컨텍스트 기반 답변 생성
- `generate_answer()`: Gradio 인터페이스용 메인 함수

**클래스 구조**:
1. LLM 초기화 (답변 생성용, 관련성 평가용)
2. 벡터 스토어 검색기 설정
3. 프롬프트 템플릿 정의
4. RAG 체인 구성

In [95]:
import gradio as gr
from langchain_core.language_models import BaseChatModel
from langchain_core.vectorstores import VectorStoreRetriever
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from typing import List, Optional, Generator
from dataclasses import dataclass

@dataclass
class SearchResult:
    context: str
    source_documents: Optional[List]

class RAGSystem:
    def __init__(
            self, 
            llm: BaseChatModel, 
            eval_llm: BaseChatModel,
            retriever: VectorStoreRetriever
        ):
        if not llm:
            self.llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
        else:
            self.llm = llm

        if not eval_llm:
            self.eval_llm = ChatOpenAI(model="gpt-4.1", temperature=0)
        else:
            self.eval_llm = eval_llm

        if not retriever:
            raise ValueError("검색기(retriever)가 필요합니다.")
        else:
            self.retriever = retriever
        
    def _format_docs(self, docs: List) -> str:
        return "\n\n".join(doc.page_content for doc in docs)
    
    def _format_source_documents(self, docs: Optional[List]) -> str:
        if not docs:
            return "\n\nℹ️ 관련 문서를 찾을 수 없습니다."
        
        formatted_docs = []
        for i, doc in enumerate(docs, 1):
            metadata = doc.metadata if hasattr(doc, 'metadata') else {}
            source_info = []
            
            if 'question_id' in metadata:
                source_info.append(f"ID: {metadata['question_id']}")
            if 'keyword' in metadata:
                source_info.append(f"키워드: {metadata['keyword']}")
            if 'summary' in metadata:
                source_info.append(f"요약: {metadata['summary']}")
                
            formatted_docs.append(
                f"📚 참조 문서 {i}\n"
                f"• {' | '.join(source_info) if source_info else '출처 정보 없음'}\n"
                f"• 내용: {doc.page_content}"
            )
        
        return "\n\n" + "\n\n".join(formatted_docs)
    
    def _check_relevance(self, docs: List, question: str) -> List:
        """문서의 관련성 확인"""

        relevant_docs = []

        if not docs:
            return relevant_docs
            
        prompt = ChatPromptTemplate.from_messages([
            ("system", """주어진 컨텍스트가 질문에 답변하는데 필요한 정보를 포함하고 있는지 평가하세요.

        다음 기준 중 하나 이상을 충족할 경우 'Yes'로 답변하고, 모두 충족하지 못하면 'No'로 답변하세요:

        1. 컨텍스트가 질문에 답변하는데 필요한 정보를 직접적으로 포함하고 있는가?
        2. 컨텍스트의 정보로부터 답변에 필요한 내용을 논리적으로 추론할 수 있는가?

        'Yes' 또는 'No'로만 답변하세요."""),
            ("human", """[컨텍스트]
        {context}

        [질문]
        {question}""")
        ])
        
        chain = prompt | self.eval_llm | StrOutputParser()

        for doc in docs:
            result = chain.invoke({
                "context": doc.page_content,
                "question": question
            }).lower()

            print(f"문서 {doc.metadata['question_id']} 관련성 확인 결과: {result}")
            print(f"문서 {doc.metadata['question_id']} 내용:")
            print(doc.page_content)
            print("-" * 50)

            if "yes" in result:
                relevant_docs.append(doc)
            
        return relevant_docs
    
    def search_documents(self, question: str) -> SearchResult:
        try:
            docs = retriever.invoke(question)
            print(f"검색된 문서 개수: {len(docs)}")
            relevant_docs = self._check_relevance(docs, question) 
            print(f"관련 문서 개수: {len(relevant_docs)}")
            
            return SearchResult(
                context=self._format_docs(relevant_docs) if relevant_docs else "관련 문서를 찾을 수 없습니다.",
                source_documents=relevant_docs,
            )
        except Exception as e:
            print(f"문서 검색 중 오류 발생: {e}")
            return SearchResult(
                context="문서 검색 중 오류가 발생했습니다.",
                source_documents=None,
            )
    
    def generate_answer(self, message: str, history: List) -> Generator[str, None, None]:
            """Gradio 스트리밍 출력을 위한 제너레이터 함수"""
            
            # 1. 문서 검색 
            search_result = self.search_documents(message)
            
            if not search_result.source_documents:
                yield "죄송합니다. 관련 문서를 찾을 수 없어 답변하기 어렵습니다. 다른 질문을 해주시겠습니까?"
                return
                        
            # 2. 프롬프트 템플릿 설정
            prompt = ChatPromptTemplate.from_messages([
                ("system", """다음 지침을 따라 질문에 답변해주세요:
                1. 주어진 문서의 내용만을 기반으로 답변하세요.
                2. 문서에 명확한 근거가 없는 내용은 "근거 없음"이라고 답변하세요.
                3. 답변하기 어려운 질문은 "잘 모르겠습니다"라고 답변하세요.
                4. 추측이나 일반적인 지식을 사용하지 마세요."""),
                ("human", "문서들:\n{context}\n\n질문: {question}")
            ])
            
            # 3. RAG Chain 구성
            chain = prompt | self.llm | StrOutputParser()
            
            full_answer = ""
            try:
                # 4. 스트리밍 실행 (chain.stream 사용)
                for chunk in chain.stream({
                    "context": search_result.context,
                    "question": message
                }):
                    full_answer += chunk
                    # 현재까지 생성된 텍스트를 Gradio UI에 즉시 반영
                    yield full_answer
                
                # 5. 답변 생성이 완료된 후 참조 문서 추가
                sources = self._format_source_documents(search_result.source_documents)
                final_response = f"{full_answer}\n\n---\n{sources}"
                yield final_response
                
            except Exception as e:
                yield f"답변 생성 중 오류가 발생했습니다: {str(e)}"

# Gradio 인터페이스 설정

rag_system = RAGSystem(
    llm=ChatOpenAI(model="gpt-4.1-nano", temperature=0),   
    eval_llm=ChatOpenAI(model="gpt-4.1-mini", temperature=0),
    retriever=vector_store.as_retriever(search_kwargs={"k": 3})
)

demo = gr.ChatInterface(
    fn=rag_system.generate_answer,
    title="RAG QA 시스템",
    description="""
    질문을 입력하면 관련 문서를 검색하여 답변을 생성합니다.
    모든 답변에는 참조한 문서의 출처가 표시됩니다.
    """,
    examples=[
        ["수원시의 주택건설지역은 어디에 해당하나요?"],
        ["무주택 세대에 대해서 설명해주세요."],
        ["2순위로 당첨된 사람이 청약통장을 다시 사용할 수 있나요?"],
    ],
)

# 데모 실행
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [96]:
# Gradio 인터페이스 종료
demo.close()

Closing server running on port: 7860


---

# **[실습] 주택청약 FAQ 시스템 구현**

### **문제 설명**
이전 코드를 기반으로 주택청약 FAQ 시스템을 다음 요구사항에 맞춰 개선합니다. 

1. 응답 품질 향상 (1개 이상)
   - 생성된 답변의 품질을 평가 (답변이 불충분한 경우 예외 처리)
   - 관련성 높은 FAQ 문서 검색 (임베딩 모델, 청크 크기, 벡터 검색 방법 등)

2. 사용자 경험 개선 (1개 이상)
   - 대화 이력 관리 기능 추가 (요약, 트리밍 기능 등 고려)
   - 최근 대화 기반 컨텍스트 구성 
   - 사용자 프로필 기반 맞춤 응답

### **제약 조건**
- Gradio ChatInterface 사용
- RAG 구조 유지

In [ ]:
import gradio as gr
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from pydantic import BaseModel, Field

# [품질 향상] 문서 관련성 평가용 스키마
class GradeAnswer(BaseModel):
    """답변의 품질 및 질문과의 관련성 평가"""
    score: str = Field(description="답변이 질문에 충분한 정보를 제공하는지 여부 ('yes' 또는 'no')")
    reason: str = Field(description="평가 이유")

# LLM 및 Retriever 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(persist_directory="./chroma_db_summary", embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={'k': 3, 'fetch_k': 10})

# 구조화된 출력기 설정
grader_llm = llm.with_structured_output(GradeAnswer)

# [사용자 경험 개선] 대화 이력을 고려한 질문 재구성 프롬프트
contextualize_q_system_prompt = """이전 대화 내용과 최신 사용자 질문이 주어졌을 때, 
이전 대화 내용 없이도 이해할 수 있는 독립적인 질문으로 바꾸세요. 질문에 답할 필요는 없습니다."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# [응답 품질 향상] 최종 답변 생성 프롬프트
system_prompt = """당신은 주택청약 전문가입니다. 제공된 컨텍스트를 사용하여 답변하세요. 
답변을 모른다면 모른다고 말하되 지어내지 마세요. 친절하게 답변하세요.

콘텐츠: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# 메모리 저장소 (세션별 관리)
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 체인 구성
def get_rag_response(user_input, session_id="default"):
    history = get_session_history(session_id)
    
    # 1. 문서 검색
    docs = retriever.invoke(user_input)
    context = "\n\n".join([d.page_content for d in docs])
    
    # 2. 답변 생성
    rag_chain = qa_prompt | llm | StrOutputParser()
    answer = rag_chain.invoke({"input": user_input, "context": context, "chat_history": history.messages})
    
    # 3. [품질 향상] 답변 검수 (Self-Correction)
    grade = grader_llm.invoke(f"질문: {user_input}\n답변: {answer}")
    
    if grade.score == "no":
        final_answer = f"⚠️ 죄송합니다. 정확한 FAQ 정보를 찾지 못했습니다. 보다 구체적으로 질문해 주시겠어요?\n(참고 사유: {grade.reason})"
    else:
        final_answer = answer
        # 4. [사용자 경험] 대화 이력 업데이트 (성공적인 답변만 저장)
        history.add_user_message(user_input)
        history.add_ai_message(final_answer)
        
    return final_answer

def predict(message, history):
    # Gradio의 history 형식을 LangChain 메시지로 변환하거나, 
    # 위에서 만든 get_rag_response의 세션 관리를 사용하여 응답 반환
    response = get_rag_response(message)
    return response

# Gradio 실행
demo = gr.ChatInterface(
    predict,
    title="🏢 스마트 주택청약 FAQ 챗봇",
    description="청약 자격, 특별공급, 소득 기준 등 궁금한 점을 물어보세요!",
    examples=["신혼부부 특별공급 조건 알려줘", "청약통장 해지하면 어떻게 돼?", "생애최초 소득 기준"]
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
